### Setup and imports

In [8]:
%pip install pandas transformers torch rapidfuzz tqdm protobuf sentencepiece dateparser -q 

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import os, re, json, dateparser, torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification, pipeline
from datetime import datetime
from dateparser.search import search_dates
from tqdm import tqdm

### Load PSGC Reference Data

In [14]:
cwd = os.getcwd()
if cwd.endswith("data_preprocessing"):
    base_dir = os.path.dirname(cwd)   # go up to project root
else:
    base_dir = cwd

# Construct paths safely
psgc_dir = os.path.join(base_dir, "data_preprocessing", "PSGC-lookup")
BRGY_CSV = os.path.join(psgc_dir, "refbrgy.csv")
CITY_CSV = os.path.join(psgc_dir, "refcitymun.csv")
REG_CSV  = os.path.join(psgc_dir, "refregion.csv")
PROV_CSV = os.path.join(psgc_dir, "refprovince.csv")

# Verify
for path in [BRGY_CSV, CITY_CSV, REG_CSV, PROV_CSV]:
    print(path, "✅ exists" if os.path.exists(path) else "❌ missing")


df_region   = pd.read_csv(REG_CSV, dtype=str).rename(columns={"regDesc": "name"})
df_province = pd.read_csv(PROV_CSV, dtype=str).rename(columns={"provDesc": "name"})
df_citymun  = pd.read_csv(CITY_CSV, dtype=str).rename(columns={"citymunDesc": "name"})
df_brgy     = pd.read_csv(BRGY_CSV, dtype=str).rename(columns={"brgyDesc": "name"})

# Normalize: lowercase + strip spaces
df_region["name_norm"]   = df_region["name"].str.lower().str.strip()
df_province["name_norm"] = df_province["name"].str.lower().str.strip()
df_citymun["name_norm"]  = df_citymun["name"].str.lower().str.strip()
df_brgy["name_norm"]     = df_brgy["name"].str.lower().str.strip()


/Users/casseygempesaw/Desktop/Thesis/disaster-scraper/data_preprocessing/PSGC-lookup/refbrgy.csv ✅ exists
/Users/casseygempesaw/Desktop/Thesis/disaster-scraper/data_preprocessing/PSGC-lookup/refcitymun.csv ✅ exists
/Users/casseygempesaw/Desktop/Thesis/disaster-scraper/data_preprocessing/PSGC-lookup/refregion.csv ✅ exists
/Users/casseygempesaw/Desktop/Thesis/disaster-scraper/data_preprocessing/PSGC-lookup/refprovince.csv ✅ exists


### Load Dataset

In [11]:
# Choose which dataset you want to load
raw_dataset     = os.path.join(base_dir, "data", "raw-data", "merged_raw_disaster_posts.csv")
cleaned_dataset = os.path.join(base_dir, "data", "cleaned-data", "cleaned_disaster_posts.csv")

# Pick raw or cleaned (set active here)
DATA_CSV = raw_dataset   # or = cleaned_dataset

# Load
if not os.path.exists(DATA_CSV):
    raise FileNotFoundError(f"❌ Dataset not found at {DATA_CSV}")
else:
    print(f"✅ Using dataset: {DATA_CSV}")

df_data = pd.read_csv(DATA_CSV)
# df_data.head()

✅ Using dataset: /Users/casseygempesaw/Desktop/Thesis/disaster-scraper/data/raw-data/merged_raw_disaster_posts.csv


### Load XLM-RoBERTa NER

This will run NER first on raw data, using the model provided by the NER pipeline. Then it will extract the locations from the NER output.

In [15]:
model_name = "Davlan/xlm-roberta-base-ner-hrl"

# 👇 sentencepiece-based tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

model = AutoModelForTokenClassification.from_pretrained(model_name)

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    device=-1,  # force CPU on M1
    aggregation_strategy="simple"
)

Device set to use cpu


#### Pre-NER slight data normalization

In [16]:
REGION_NUM_RE = re.compile(r"\bregion\s+(\d{1,2}|[ivx]{1,4})\b", re.I)
PAREN_NAME_RE = re.compile(r"\(([^)]+region)\)", re.I)
CAR_RE = re.compile(r"\bCAR\b", re.I)   # Cordillera Administrative Region

# ---- Hashtag normalizer (run BEFORE NER) ----
def _split_camel(s: str) -> str:
    # insert spaces between lower→Upper (e.g., "DavaoDeOro" -> "Davao De Oro")
    s = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', s)
    # split letters↔digits boundaries if any
    s = re.sub(r'([A-Za-z])(\d)', r'\1 \2', s)
    s = re.sub(r'(\d)([A-Za-z])', r'\1 \2', s)
    return s

def normalize_hashtags_in_text(text: str) -> str:
    # Replace hashtags with spaced tokens (drop '#')
    def _repl(m):
        tag = m.group(1)
        # Handle snake_case / kebab-case
        tag = tag.replace('_', ' ').replace('-', ' ')
        # Camel/Pascal case split
        tag = _split_camel(tag)
        return tag
    return re.sub(r"#([A-Za-z0-9_\-]+)", _repl, text)

# (optional) general text preprocessor hook
def _preprocess_for_ner(text: str) -> str:
    t = normalize_hashtags_in_text(text)
    # you can add other light normalizations here if needed
    return t

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).lower().strip())

# Add normalized columns (if not yet added)
for _df in (df_region, df_province, df_citymun, df_brgy):
    _df["name_norm"] = _df["name"].map(_norm)

def extract_region_hints(text: str):
    hints = []

    # Region number (Region 1, Region XI)
    for m in REGION_NUM_RE.finditer(text):
        hints.append(_norm(f"region {m.group(1)}"))

    # Names inside parentheses: (Ilocos Region), (Central Luzon)
    for m in PAREN_NAME_RE.finditer(text):
        hints.append(_norm(m.group(1)))

    # Special acronyms
    if CAR_RE.search(text):
        hints.append("cordillera administrative region")

    return list(dict.fromkeys(hints))  # de-dup


# Clean entities (merge entities of same group)
def clean_entities(entities):
    merged = []
    buffer = ""
    last_group = None
    
    for ent in entities:
        word = ent["word"].replace("##", "").strip()
        if ent["entity_group"] == last_group:
            buffer += word
        else:
            if buffer:
                merged.append(_norm(buffer))  # normalize here
            buffer = word
            last_group = ent["entity_group"]
    if buffer:
        merged.append(_norm(buffer))
    return merged

def extract_entities_xlmr(text: str):
    pre = _preprocess_for_ner(str(text))
    entities = ner_pipeline(pre)

    spans = []
    for ent in entities:
        if ent.get("entity_group") in ["LOC", "GPE"]:
            spans.append(_norm(ent["word"]))

    # Add region hints
    spans += extract_region_hints(pre)

    # De-dup preserve order
    seen, out = set(), []
    for s in spans:
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out

## PSGC Fuzzy Matching / Helpers

This will help in structuring and getting resolved location with its full hierarchy (e.g., if Matina Pangi → add {barangay: "Matina Pangi", city: "Davao City", province: "Davao del Sur", region: "Region XI"}

#### Constants

In [ ]:
ISLAND_GROUPS = {"luzon", "visayas", "mindanao", "philippines"}

# Generic “do-not-PSGC-match” terms (treat as island groups or broad geographies)
BLOCKLIST = {
    "philippine",
    "philippine sea", "west philippine sea",
    "northern luzon", "southern luzon", "eastern visayas", "western visayas", "central visayas"
}

# Aliases → official PSGC names (normalize keys!)
ALIASES = {
    _norm("metro manila"): _norm("National Capital Region (NCR)"),
    _norm("ncr"):          _norm("National Capital Region (NCR)"),
    _norm("manila metro"): _norm("National Capital Region (NCR)"),
    _norm("maynila"): _norm("National Capital Region (NCR)"),
    _norm("car"):          _norm("Cordillera Administrative Region (CAR)"),
    _norm("barmm"):        _norm("Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)"),
    _norm("armm"):         _norm("Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)"), 
    _norm("soccsksargen"): _norm("SOCCSKSARGEN Region"),
    _norm("region 12"):    _norm("SOCCSKSARGEN Region"),
    _norm("region 11"):    _norm("Davao Region"),
    _norm("region 4a"):    _norm("CALABARZON"),
    _norm("calabarzon"):   _norm("CALABARZON"),
    _norm("region 4b"):    _norm("MIMAROPA"),
    _norm("mimaropa"):     _norm("MIMAROPA"),
    _norm("region 10"):    _norm("Northern Mindanao"),
    _norm("davao"):        _norm("Davao City")
}

ROMANS = {
    1:"i",2:"ii",3:"iii",4:"iv",5:"v",6:"vi",7:"vii",8:"viii",9:"ix",10:"x",
    11:"xi",12:"xii",13:"xiii",14:"xiv",15:"xv",16:"xvi",17:"xvii",18:"xviii"
}

# POI / landmark words to strip
POI_TERMS = {
    "road","rd","street","st","ave","avenue","blvd","highway","hwy",
    "bridge","overpass","underpass","rotunda","roundabout",
    "market","terminal","airport","pier","port","wharf","harbor",
    "school","church","chapel","cemetery","hospital","clinic",
    "mall","plaza","park","grounds","field","gym","court",
    "river","creek","canal",
    "subdivision","subd","village","vlge","phase","blk","block",
    "barangay hall","brgy hall", "bgy.", "bgy", "brgy", "barangay", "brgy.", "volcano",
    "mount", "mountain"
}

DIRECTIONS = {
    # English
    "northern", "southern", "eastern", "western",
    "north", "south", "east", "west",

    # Filipino
    "hilaga", "hilagang",   # north
    "timog", "timogang",    # south
    "silangan", "silangang",# east
    "kanluran", "kanlurang",# west

    # Weather / directional winds (often used)
    "amihan",   # northeast tradewind
    "habagat",  # southwest tradewind
    "habagatan" # southward
}


#### Helper functions

In [ ]:
# ================================
#  Normalize Island Groups
# ================================
def normalize_direction_island(e: str):
    """
    If text is 'northern luzon' → return 'luzon'
    If text is 'eastern visayas' → return 'visayas'
    Else return None
    """
    parts = e.split()
    if len(parts) == 2 and parts[0] in DIRECTIONS and parts[1] in ISLAND_GROUPS:
        return parts[1]   # strip direction, keep island group
    return None

# ================================
#  Remove POI/landmark terms from a phrase, keep core toponym
# ================================
def strip_poi_terms(s: str) -> str:
    txt = _norm(s)
    txt = re.sub(r"[^a-z0-9\s\-]", " ", txt)  # soften punctuation
    # multi-word first
    for phrase in sorted([p for p in POI_TERMS if " " in p], key=len, reverse=True):
        txt = re.sub(rf"\b{re.escape(phrase)}\b", " ", txt)
    # single-word
    for word in [p for p in POI_TERMS if " " not in p]:
        txt = re.sub(rf"\b{re.escape(word)}\b", " ", txt)
    return re.sub(r"\s+", " ", txt).strip()

# ================================
#   Parenthesis -- to match places with ()
# ================================
def _extract_paren_name(name):
    # e.g. "Region XI (Davao Region)" -> "Davao Region"
    m = re.search(r"\(([^)]+)\)", name)
    return m.group(1) if m else None

region_alias_map = {}  # alias_norm -> canonical region name_norm
for _, row in df_region.iterrows():
    official = row["name"]                # PSGC official name string
    official_norm = _norm(official)
    paren = _extract_paren_name(official)
    paren_norm = _norm(paren) if paren else None

    # Add official name
    region_alias_map[official_norm] = official_norm
    if paren_norm:
        region_alias_map[paren_norm] = official_norm

    # From regCode, generate "region 11" and "region xi" aliases when possible
    # regCode might be like "110000000" or "11"
    code_digits = re.search(r"(\d+)", "".join(ch for ch in str(row.get("regCode","")) if ch.isdigit()))
    if code_digits:
        n = int(code_digits.group(1)[:2])  # grab first 1–2 digits
        if 1 <= n <= 20:
            region_alias_map[_norm(f"region {n}")] = official_norm
            region_alias_map[_norm(f"region {ROMANS.get(n,'')}")] = official_norm
            
# merge into region_alias_map (only if target exists in df_region)
for k, v in ALIASES.items():
    if v in region_alias_map.values():
        region_alias_map[k] = v

# ================================
#   IGNORE >= 4 character tokens
# ================================
def _looks_like_junk(token_norm: str) -> bool:
    # drop very short fragments unless they’re exact PSGC names
    return len(token_norm) < 4

# ================================
#       Try both 'X City' <-> 'City of X'
# ================================
def try_city_variants(e: str, raw: str):
    variants = set()

    # If it's 'X city' -> also try 'city of X'
    m = re.match(r"^(.*)\s+city$", e)
    if m:
        base = m.group(1).strip()
        variants.add(f"city of {base}")

    # If it's 'city of X' -> also try 'X city'
    m = re.match(r"^city of\s+(.*)$", e)
    if m:
        base = m.group(1).strip()
        variants.add(f"{base} city")

    # Also try straight pass-throughs if user gave just 'malabon' (handled earlier),
    # but include here as a safety net: generate both forms from a bare base
    # (this will be no-op if e already has 'city' wording).
    if "city" not in e:
        variants.add(f"{e} city")
        variants.add(f"city of {e}")

    # Try the variants against city/municipality table
    for v in variants:
        alias_norm = _norm(v)
        hit = df_citymun[df_citymun["name_norm"] == alias_norm]
        if not hit.empty:
            r = hit.iloc[0]
            col = next(c for c in r.index if "citymun" in c.lower() and "code" in c.lower())
            return {"entity": raw, "level": "citymun", "code": r[col], "matched_name": r["name"]}

    return None


# ================================
#  Helpers for PSGC code columns
# ================================
def _get_code_col(df, key):
    return next(c for c in df.columns if key in c.lower() and "code" in c.lower())

BRGY_CODE_COL    = _get_code_col(df_brgy, "brgy")
BRGY_CITY_COL    = _get_code_col(df_brgy, "citymun")
CITYMUN_CODE_COL = _get_code_col(df_citymun, "citymun")
PROV_CODE_COL    = _get_code_col(df_province, "prov")
REG_CODE_COL     = _get_code_col(df_region, "reg")


# ================================
# Matching with priority: region > province > city/mun > barangay
# ================================
def match_psgc(entity: str, context_city_code: str | None = None):
    raw = str(entity)
    e = _norm(raw)

    # 0) Direction + island → island group
    norm_island = normalize_direction_island(e)
    if norm_island:
        return {"entity": raw, "level": "island_group", "code": None, "matched_name": norm_island.title()}

    # 1) Non-PSGC labels / blocks
    if e in ISLAND_GROUPS:
        return {"entity": raw, "level": "island_group", "code": None, "matched_name": raw}
    if e in BLOCKLIST:
        return {"entity": raw, "level": None, "code": None, "matched_name": None}

    # 2) Region aliases
    if e in region_alias_map:
        target_norm = region_alias_map[e]
        hit = df_region[df_region["name_norm"] == target_norm]
        if not hit.empty:
            r = hit.iloc[0]
            return {"entity": raw, "level": "region", "code": r[REG_CODE_COL], "matched_name": r["name"]}

    # 3) General aliases (region/province/city/brgy by name_norm)
    if e in ALIASES:
        alias_norm = ALIASES[e]
        hit = df_region[df_region["name_norm"] == alias_norm]
        if not hit.empty:
            r = hit.iloc[0]
            return {"entity": raw, "level": "region", "code": r[REG_CODE_COL], "matched_name": r["name"]}
        hit = df_province[df_province["name_norm"] == alias_norm]
        if not hit.empty:
            r = hit.iloc[0]
            return {"entity": raw, "level": "province", "code": r[PROV_CODE_COL], "matched_name": r["name"]}
        hit = df_citymun[df_citymun["name_norm"] == alias_norm]
        if not hit.empty:
            r = hit.iloc[0]
            return {"entity": raw, "level": "citymun", "code": r[CITYMUN_CODE_COL], "matched_name": r["name"]}
        hit = df_brgy[df_brgy["name_norm"] == alias_norm]
        if not hit.empty:
            r = hit.iloc[0]
            return {"entity": raw, "level": "barangay", "code": r[BRGY_CODE_COL], "matched_name": r["name"]}

    # 4) Pattern-based city expansions
    result = try_city_variants(e, raw)
    if result:
        return result

    # 5) Exact matches by priority (province > city/mun > barangay)
    hit = df_province[df_province["name_norm"] == e]
    if not hit.empty:
        r = hit.iloc[0]
        return {"entity": raw, "level": "province", "code": r[PROV_CODE_COL], "matched_name": r["name"]}

    hit = df_citymun[df_citymun["name_norm"] == e]
    if not hit.empty:
        r = hit.iloc[0]
        return {"entity": raw, "level": "citymun", "code": r[CITYMUN_CODE_COL], "matched_name": r["name"]}

    if not _looks_like_junk(e):
        hit = df_brgy[df_brgy["name_norm"] == e]
        if not hit.empty:
            r = hit.iloc[0]
            return {"entity": raw, "level": "barangay", "code": r[BRGY_CODE_COL], "matched_name": r["name"]}

    # 6) POI stripping fallback (e.g., 'matina pangi road' -> 'matina pangi')
    e_clean = strip_poi_terms(e)
    if e_clean and e_clean != e:
        # try again with cleaned phrase (repeat steps 4–5 minimal)
        hit = df_citymun[df_citymun["name_norm"] == e_clean]
        if not hit.empty:
            r = hit.iloc[0]
            return {"entity": raw, "level": "citymun", "code": r[CITYMUN_CODE_COL], "matched_name": r["name"]}

        if not _looks_like_junk(e_clean):
            # if a city context exists, restrict barangays to that city
            if context_city_code:
                hit = df_brgy[(df_brgy["name_norm"] == e_clean) &
                              (df_brgy[BRGY_CITY_COL].astype(str).str.startswith(str(context_city_code)))]
                if not hit.empty:
                    r = hit.iloc[0]
                    return {"entity": raw, "level": "barangay", "code": r[BRGY_CODE_COL], "matched_name": r["name"]}
            # else try any barangay with that name
            hit = df_brgy[df_brgy["name_norm"] == e_clean]
            if not hit.empty:
                r = hit.iloc[0]
                return {"entity": raw, "level": "barangay", "code": r[BRGY_CODE_COL], "matched_name": r["name"]}

    # 7) Nothing reliable
    return {"entity": raw, "level": None, "code": None, "matched_name": None}


## DATE TIME Extraction

In [ ]:
PH_TZ = "Asia/Manila"

MONTH_MAP = {
    # Filipino full
    r"\benero\b": "january",    r"\bpebrero\b": "february", r"\bpebro\b": "february",
    r"\bpebr\b": "feb",         r"\bpeb\b": "feb",
    r"\bmarso\b": "march",      r"\babril\b": "april",      r"\bmayo\b": "may",
    r"\bhunyo\b": "june",       r"\bhulyo\b": "july",       r"\bagosto\b": "august",
    r"\bsetyembre\b": "september", r"\boktubre\b": "october",
    r"\bnobyembre\b": "november",  r"\bdisyembre\b": "december",

    # Filipino short (common)
    r"\bhun\b": "jun", r"\bhul\b": "jul", r"\bago\b": "aug",
    r"\bset\b": "sep", r"\bokt\b": "oct", r"\bnob\b": "nov", r"\bdis\b": "dec",
}

TEMPORAL_WORD_MAP = {
    r"\bkanina\b": "earlier", r"\bkahapon\b": "yesterday", r"\bngayon\b": "today",
    r"\bmamaya\b": "later",   r"\bbukas\b": "tomorrow",
    r"\bumaga\b": "morning",  r"\btanghali\b": "noon",     r"\bhapon\b": "afternoon",
    r"\bgabi\b": "evening",   r"\bhatinggabi\b": "midnight",
    # Cebuano
    r"\bgahapon\b": "yesterday", r"\bkaron\b": "today", r"\bugma\b": "tomorrow",
    r"\bbuntag\b": "morning",    r"\budto\b": "noon",  r"\bgabii\b": "evening",
}

DATE_PATTERNS = [
    r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\.?\s+\d{1,2}(?:,\s*\d{2,4})?\b",
    r"\b\d{1,2}/\d{1,2}(?:/\d{2,4})?\b",
    r"\b\d{4}-\d{2}-\d{2}(?:[ T]\d{1,2}:\d{2}(?::\d{2})?)?\b",
    r"\b(?:today|tonight|yesterday|tomorrow)\b",
    r"\b(?:kanina|kahapon|ngayon|mamaya|bukas)\b",
    r"\b(?:umaga|tanghali|hapon|gabi|hatinggabi)\b",
    r"\b(?:\d{1,2})(?:[:.]\d{2})?\s*(?:am|pm)\b",
    r"\b(?:noon|midnight)\b",
]
DATE_RE = re.compile("|".join(DATE_PATTERNS), flags=re.IGNORECASE)

# 🔹 explicit time tokens (12h/24h)
TIME_TOKEN_RE = re.compile(
    r"\b(?:[01]?\d|2[0-3])[:.][0-5]\d(?:[:.][0-5]\d)?\b|\b\d{1,2}\s*(?:am|pm)\b|\bnoon\b|\bmidnight\b",
    flags=re.IGNORECASE,
)

def _pre_norm_temporal(text: str) -> str:
    t = text
    for pat, repl in {**MONTH_MAP, **TEMPORAL_WORD_MAP}.items():
        t = re.sub(pat, repl, t, flags=re.IGNORECASE)
    return t

def _first_present(cols, row):
    for c in cols:
        if c in row and pd.notna(row[c]) and str(row[c]).strip():
            return row[c]
    return None

def _parse_ref_dt(raw, default_now=True):
    if raw is None:
        return datetime.now()
    dt = dateparser.parse(str(raw), settings={"TIMEZONE": PH_TZ, "RETURN_AS_TIMEZONE_AWARE": False})
    if dt is None and default_now:
        dt = datetime.now()
    return dt

def extract_temporal_mentions(text: str):
    if not text:
        return []
    t = _pre_norm_temporal(text)
    hits = [m.group(0) for m in DATE_RE.finditer(t)]
    try:
        sd = search_dates(
            t,
            languages=["en", "tl"],
            settings={"PREFER_DATES_FROM": "past", "TIMEZONE": PH_TZ, "RETURN_AS_TIMEZONE_AWARE": False}
        ) or []
    except ValueError:
        sd = search_dates(
            t,
            languages=["en"],
            settings={"PREFER_DATES_FROM": "past", "TIMEZONE": PH_TZ, "RETURN_AS_TIMEZONE_AWARE": False}
        ) or []
    hits += [s for (s, _) in sd]
    seen, out = set(), []
    for h in hits:
        k = h.lower().strip()
        if k and k not in seen:
            seen.add(k)
            out.append(h.strip())
    return out

def normalize_temporal(text: str, ref_dt: datetime):
    if not text:
        return []
    t = _pre_norm_temporal(text)
    try:
        results = search_dates(
            t,
            languages=["en", "tl"],
            settings={
                "PREFER_DATES_FROM": "past",
                "RELATIVE_BASE": ref_dt,
                "TIMEZONE": PH_TZ,
                "RETURN_AS_TIMEZONE_AWARE": False,
            },
        ) or []
    except ValueError:
        results = search_dates(
            t,
            languages=["en"],
            settings={
                "PREFER_DATES_FROM": "past",
                "RELATIVE_BASE": ref_dt,
                "TIMEZONE": PH_TZ,
                "RETURN_AS_TIMEZONE_AWARE": False,
            },
        ) or []
    out, seen = [], set()
    for mention, dt in results:
        iso = dt.isoformat()
        if iso not in seen:
            seen.add(iso)
            out.append({"mention": mention, "iso": iso})
    return out

# 🔹 helpers to pick best primary time (text-first, prefer time, merge time-only if needed)
def _has_time_in_mention(mention: str) -> bool:
    return bool(TIME_TOKEN_RE.search(mention or ""))

def _has_time_in_iso(iso_str: str) -> bool:
    try:
        dt = datetime.fromisoformat(iso_str)
        return (dt.hour, dt.minute, dt.second) != (0, 0, 0)
    except Exception:
        return False

def _extract_time_only_mentions(text: str):
    """Return first time-of-day mention (e.g., '6PM', '18:30') if present."""
    t = _pre_norm_temporal(text)
    m = TIME_TOKEN_RE.search(t or "")
    return m.group(0) if m else None

def _overlay_time_on_date(date_iso: str, time_str: str, ref_dt: datetime) -> str | None:
    """
    Combine a date-only ISO with a separate time string.
    We parse time_str using the same date as RELATIVE_BASE so only the time parts are taken.
    """
    if not date_iso or not time_str:
        return None
    try:
        base = datetime.fromisoformat(date_iso)
    except Exception:
        return None
    parsed_time = dateparser.parse(
        time_str,
        settings={
            "RELATIVE_BASE": base,
            "TIMEZONE": PH_TZ,
            "RETURN_AS_TIMEZONE_AWARE": False,
            "PREFER_DATES_FROM": "past",
        },
    )
    if not parsed_time:
        return None
    # keep the date from base, replace time from parsed_time
    combined = base.replace(hour=parsed_time.hour, minute=parsed_time.minute, second=parsed_time.second or 0)
    return combined.isoformat()

def pick_primary_event_time(parsed_list, raw_text: str, ref_dt_fallback_iso: str | None):
    """
    Priority:
    1) First parsed item that has a time (by mention or ISO)
    2) Else first parsed date; if no time, try merge with a separate time-only mention
    3) Else fallback to metadata timestamp (post/scraped)
    """
    if parsed_list:
        # 1) prefer items with time
        for p in parsed_list:
            if _has_time_in_mention(p.get("mention", "")) or _has_time_in_iso(p.get("iso", "")):
                return p["iso"]

        # 2) take first date; try to merge time-only mention
        first_iso = parsed_list[0]["iso"]
        if not _has_time_in_iso(first_iso):
            t_only = _extract_time_only_mentions(raw_text)
            merged = _overlay_time_on_date(first_iso, t_only, datetime.now()) if t_only else None
            if merged:
                return merged
        return first_iso

    # 3) fallback
    return ref_dt_fallback_iso

## Sentiment Score

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SENT_MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

sent_tok = AutoTokenizer.from_pretrained(SENT_MODEL)
sent_mdl = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL).to(DEVICE)
id2label = {int(k): v for k, v in sent_mdl.config.id2label.items()}

def _find_ids(name_parts):
    return [i for i, lab in id2label.items() if any(p in lab.lower() for p in name_parts)]

pos_ids = _find_ids(["pos"]) or [2]
neg_ids = _find_ids(["neg"]) or [0]
neu_ids = _find_ids(["neu"]) or [1]

def clip(s, n=2000): return s[:n] if isinstance(s,str) else ""

texts = [clip(t) for t in df_data["text_content"].astype(str).tolist()]

scores, labels = [], []
B = 32
for i in range(0, len(texts), B):
    batch = texts[i:i+B]
    enc = sent_tok(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        logits = sent_mdl(**enc).logits
        probs = logits.softmax(-1).cpu().numpy()  # [B,3]
    pred = probs.argmax(-1)
    labels.extend([id2label[int(j)].lower() for j in pred])
    # signed score in [-1, 1]
    s = probs[:, pos_ids].sum(axis=1) - probs[:, neg_ids].sum(axis=1)
    scores.extend(s.tolist())

df_data["sentiment_label"] = labels
df_data["sentiment_score"] = scores
print(pd.Series(df_data["sentiment_label"]).value_counts())
df_data["sentiment_score"].describe()

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

NameError: name 'df_data' is not defined

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

### Severity Label

In [ ]:
sent_to_sev = {
    "negative": "major",
    "neutral": "moderate",
    "positive": "minor"
}

df_data["severity_label"] = df_data["sentiment_label"].map(sent_to_sev).fillna("moderate")  # fallback = moderate
df_data["severity_level"] = df_data["severity_label"].map({
    "minor": 1,
    "moderate": 2,
    "major": 3
}).astype("Int8")

print(df_data[["sentiment_label", "sentiment_score", "severity_label", "severity_level"]].head())
print("Severity counts:\n", df_data["severity_label"].value_counts())

## Disaster Type

In [ ]:
# 1) Volcano name list (with variants/typos seen in posts)
PH_VOLCANOES = [
    "Mayon","Taal","Kanlaon","Bulusan","Pinatubo","Hibok Hibok","Hibok-Hibok",
    "Banahaw","Banahao","Isarog","Iraya","Matutum","Ragang","Didicas","Babuyan Claro",
    "Smith","Parker","Apo","Musuan","Makaturing","Leonard Kniaseff","Biliran",
    "Camiguin","Balusan"
]
def _vname_pat(n):
    s = re.escape(n)
    s = s.replace(r"\ ", r"\s+").replace(r"\-", r"[-\s]?")
    return s
VOLCANO_NAME_RE = r"(?:%s)" % "|".join(_vname_pat(n) for n in PH_VOLCANOES)

# 2) Extra local/hashtag patterns derived from your scraping KEYWORDS
EXTRA = {
    "flood": [
        r"\blunop\b", r"\bpaglunop\b", r"\bnabahaan?\b", r"\bstreet\s*flood\b",
        r"\b(?:(?:river|creek|ilog)\s+)?overflow\b", r"#flood", r"#floodalert", r"#bahaph", r"#streetfloodalert"
    ],
    "landslide": [
        r"\bnahulog\s+ang\s+yuta\b", r"\bslope\s+failure\b", r"#landslide", r"#landslidealert", r"#landslideph"
    ],
    "earthquake": [
        r"\blinog\b", r"\bnangurog\b", r"#earthquakealert", r"#earthquakeph",
        r"\bground\s+shak(?:e|ing)\b", r"\bseismic\b"
    ],
    "tsunami": [ r"\btidal\s+wave\b", r"\bsea\s+surge\b", r"\bcoastal\s+surge\b", r"#tsunami" ],
    "fire": [ r"\bnasunog\b", r"\bapoy\b", r"#firealert", r"\bwild\s*fire\b" ],
    "volcanic": [ r"\bbulkan\b", r"\bpagputok\s+ng\s+bulkan\b", r"#volcano" ],
    "monsoon_rain": [
        r"\bheavy\s+rainshowers?\b", r"\bmalakas\s+na\s+ulan\b", r"\bLPA\b", r"\blow\s+pressure\s+area\b"
    ],
    "typhoon": [ r"\bunos\b", r"\bmalakas\s+na\s+bagyo\b", r"\bmalakas\s+na\s+hangin\b", r"#typhoon", r"#storm" ],
    "general": [
        r"\bkalamidad\b", r"\bsakuna\b", r"\bdisaster\b", r"\bemergency\b",
        r"\b(rescue|relief|evacuate|evacuation|evacuated)\b", r"#evacuation", r"#warning", r"#alert"
    ]
}

# 3) Regex buckets (original + extras)
P = {
  "flood": [
    r"\bflood(?:s|ed|ing)?\b", r"\bflash[-\s]?floods?\b", r"\bbaha\b", r"\bpagbaha\b",
    r"\binundat(?:e|ed|ion)\b", r"\boverflow\b", r"\broads?\s+(?:are\s+)?impassable\b",
  ] + EXTRA["flood"],
  "typhoon": [
    r"\b(?:super)?typhoon\b", r"\btropical\s+(?:storm|depression)\b",
    r"\btropical\s+cyclone\b", r"\bcyclone\b", r"\bbagyo\b",
    r"\bwind\s+signals?\b", r"\beye\s+(?:of|near)\b",
  ] + EXTRA["typhoon"],
  "monsoon_rain": [
    r"\bmonsoon\b", r"\bhabagat\b", r"\bamihan\b", r"\bheavy\s+rains?\b",
    r"(?:\blow\s+pressure\s+area\b|\bLPA\b)",
  ] + EXTRA["monsoon_rain"],
  "storm_surge": [ r"\bstorm\s+surge\b", r"\bdaluyong\b" ],
  "landslide": [
    r"\blandslides?\b", r"\bmudslides?\b", r"\brockfalls?\b",
    r"\bpagguho\s+ng\s+lupa\b", r"\bsoil\s+erosion\b",
  ] + EXTRA["landslide"],
  "earthquake": [
    r"\bearthquakes?\b", r"\blindol\b", r"\baftershocks?\b",
    r"\bmagnitude\s*\d+(?:\.\d+)?\b", r"\bepicen(?:t|tr)e?r\b",
  ] + EXTRA["earthquake"],
  "volcanic": [
    r"\bvolcan(?:ic|o)\b", r"\beruption\b", r"\bashfall\b", r"\blahar\b",
    r"\balert\s+level\s*[1-5]\b", r"\bphivolcs\b", r"\bpyroclastic\b",
    rf"\b(?:mt\.?|mount|bulkang?|bundok)\s+{VOLCANO_NAME_RE}\b",
    rf"\b{VOLCANO_NAME_RE}\s+(?:volcano|caldera)\b",
    r"\blava\s+(?:flow|fountain|effusion)\b",
    r"\b(?:phreatic|phreatomagmatic)\b",
    r"\b(?:ash(?:fall| plume| cloud)|volcanic plume|steam emission)\b",
    r"\b(?:sulphur|sulfur)\s+dioxide\b|\bso2\b",
    r"\bvolcanic\s+(?:tremor|quake|seismicity)\b",
  ] + EXTRA["volcanic"],
  "tsunami": [ r"\btsunami\b", r"\bsea\s+level\s+(?:warning|advisory)\b" ] + EXTRA["tsunami"],
  "fire": [
    r"\b(?:building|forest|grass)?\s*fire\b", r"\bsunog\b",
    r"\b(?:\d(?:st|nd|rd|th)\s+)?alarm\b", r"\bblaze\b",
  ] + EXTRA["fire"],
  "drought_heat": [ r"\bdrought\b", r"\btagtuyot\b", r"\bel\s*niñ?o\b", r"\bheat\s+(?:wave|index)\b" ],
  "oil_spill": [ r"\boil[-\s]?spill\b", r"\b(?:tanker|barge).*(?:leak|spill)\b" ],
  "gas_chemical": [ r"\b(?:ammonia|chlorine|lpg|gas).*(?:leak|spill)\b", r"\bhazmat\b" ],
  "thunderstorm": [
    r"\bthunderstorms?\b",
    r"\b(?:severe\s+)?thunderstorm\s+advisory\b",
    r"\brain[-\s]?showers?\b", r"\blightning\b", r"\bthunder\b",
    r"\b(?:strong|gusty)\s+winds?\b", r"\bgusts?\b",
  ],
  "rainfall_warning": [
    r"\b(?:yellow|orange|red)\s+rainfall\s+warning\b",
    r"\brainfall\s+advisory\b"
  ],
}

# 4) Negation/hedging
NEG = {
  "flood": [ r"\bno\s+floods?\b", r"\bwalang\s+pagbaha\b", r"\bnot\s+(?:flooded|flooding)\b" ],
  "typhoon": [ r"\bnot\s+seen\s+to\s+make\s+landfall\b", r"\bhindi\s+(?:tatama|magla-landfall)\b" ],
  "monsoon_rain": [ r"\bimproving\s+weather\b", r"\bweak(?:ening)?\s+monsoon\b" ],
  "landslide": [ r"\bno\s+landslides?\b" ],
  "earthquake": [ r"\bno\s+(?:felt\s+)?aftershocks?\b" ],
  "fire": [ r"\bfire\s+(?:out|contained)\b" ],
  "volcanic": [ r"\bno\s+(?:eruption|ashfall|increase\s+in\s+activity)\b", r"\balert\s+level\s+(?:lowered|decreased)\b" ],
  "thunderstorm": [ r"\bthunderstorm\s+advisory\s+(?:lifted|cancelled)\b", r"\bno\s+thunderstorm\b" ],
}

PRIORITY = [
  "tsunami","volcanic","earthquake","landslide",
  "typhoon","thunderstorm","storm_surge","monsoon_rain","flood",
  "rainfall_warning","oil_spill","gas_chemical","fire","drought_heat"
]

# 5) Compile once
_compiled    = {k: [re.compile(p, re.I) for p in v] for k, v in P.items()}
_compiled_neg= {k: [re.compile(p, re.I) for p in v] for k, v in NEG.items()}

# 6) Keyword→category fallback
_fallback = {
    "flood": ["flood","flooding","flash flood","river overflow","street flood","baha","lunop","pagbaha","paglunop","#flood","#floodalert","#bahaph","#streetfloodalert"],
    "landslide": ["landslide","mudslide","soil erosion","slope failure","pagguho ng lupa","nahulog ang yuta","#landslide","#landslidealert","#landslideph"],
    "earthquake": ["earthquake","aftershock","ground shaking","seismic","lindol","linog","pagyanig","nangurog","#earthquakealert","#earthquakeph"],
    "tsunami": ["tsunami","tidal wave","sea surge","coastal surge","daluyong","#tsunami"],
    "fire": ["fire","blaze","burning","wildfire","sunog","nasunog","apoy","#firealert"],
    "volcanic": ["volcano","volcanic","eruption","ashfall","lava","bulkan","pagputok ng bulkan","#volcano","taal","mayon","pinatubo","kanlaon","bulusan","balusan"],
    "typhoon": ["typhoon","storm","tropical storm","tropical depression","super typhoon","tropical cyclone","bagyo","unos","thunderstorm","lightning","#typhoon","#storm"],
    "monsoon_rain": ["rain","raining","rainfall","heavy rain","heavy rainshowers","monsoon","lpa","low pressure area","malakas na ulan"],
    "thunderstorm": ["thunderstorm","thunder","lightning","gusty winds","gusts","rainshowers"],
    "rainfall_warning": ["yellow rainfall warning","orange rainfall warning","red rainfall warning","rainfall advisory"],
    "oil_spill": ["oil spill","tanker","barge","leak"],
    "gas_chemical": ["ammonia","chlorine","lpg","gas","hazmat"],
}

def classify_disaster(text: str, threshold: float = 1.0):
    """Return (primary, active_list, raw_scores_json).
       Falls back to keyword scan if strict regex misses.
    """
    t = str(text or "")
    scores = {}
    for cat, regs in _compiled.items():
        pos = sum(1 for rgx in regs if rgx.search(t))
        neg = sum(1 for rgx in _compiled_neg.get(cat, []) if rgx.search(t))
        s = max(0.0, pos - 0.5*neg)
        scores[cat] = s

    active = [c for c, s in scores.items() if s >= threshold]

    if not active:
        # fallback A: any "flood" tokens? force that
        if any(rgx.search(t) for rgx in _compiled["flood"]):
            active = ["flood"]
        else:
            # fallback B: keyword→category scan (broad)
            lo = t.lower()
            for cat, kws in _fallback.items():
                if any(kw in lo for kw in kws):
                    active = [cat]
                    break

            if not active:
                return "unknown", [], json.dumps(scores, ensure_ascii=False)

    # choose primary by score, then PRIORITY tie-break
    max_s = max(scores[c] for c in active) if active else 0
    top = [c for c in active if scores[c] == max_s] if active else []
    primary = (top or active)[0]
    if len(top) > 1:
        top.sort(key=lambda c: PRIORITY.index(c) if c in PRIORITY else 999)
        primary = top[0]

    return primary, active, json.dumps(scores, ensure_ascii=False)

# Choose the text column robustly (keeps your original column if it exists)
_text_cols_pref = ["text_content","text","clean_text","content","post_text","full_text","body"]
_text_col = next((c for c in _text_cols_pref if c in df_data.columns), None)
if _text_col is None:
    # fallback: first object dtype col
    _text_col = next((c for c in df_data.columns if df_data[c].dtype == "object"), df_data.columns[0])

# Apply classifier
prim, all_types, raw_scores = [], [], []
for txt in df_data[_text_col].astype(str).tolist():
    p, cats, sc = classify_disaster(txt, threshold=1.0)
    prim.append(p)
    all_types.append("; ".join(cats))
    raw_scores.append(sc)

df_data["disaster_type"]        = prim
df_data["disaster_types_all"]   = all_types
df_data["disaster_type_scores"] = raw_scores

# Quick sanity report in notebook
print("UNKNOWN rate: {:.2f}%".format(100*df_data["disaster_type"].eq("unknown").mean()))
print(df_data["disaster_type"].value_counts().head(10))


## Apply to Dataset

In [ ]:
TEXT_COL = "text"
REF_COLS = ["post_timestamp", "scraped_timestamp", "date-time", "created_at", "post_created_at"]

# ---------- PHASE 1: locations ----------
all_entities, matched = [], []

for _, row in tqdm(df_data.iterrows(), total=len(df_data), desc="Phase 1: locations"):
    text = str(row.get(TEXT_COL, "") or "")

    # NER → PSGC (with city context)
    ents = extract_entities_xlmr(text)
    rough = [match_psgc(ent, context_city_code=None) for ent in ents]
    context_city_code = next((m["code"] for m in rough if m.get("level") == "citymun" and m.get("code")), None)
    matches = [match_psgc(ent, context_city_code=context_city_code) for ent in ents]

    all_entities.append(ents)
    matched.append(matches)

df_data["location_raw"] = all_entities
df_data["location_matched"] = matched

# keep entities unless BOTH level and code are None; dedupe by (level, code)
def clean_matches(matches):
    seen, cleaned = set(), []
    for m in matches or []:
        if not m: 
            continue
        if m.get("level") is None and m.get("code") is None:
            continue
        key = (m.get("level"), m.get("code"))
        if key not in seen:
            seen.add(key)
            cleaned.append(m)
    return cleaned

df_data["location_matched_clean"] = df_data["location_matched"].apply(clean_matches)
df_data["location_matched_json"]  = df_data["location_matched_clean"].apply(
    lambda x: json.dumps(x or [], ensure_ascii=False)
)

# ---------- DROP rows with no valid locations ----------
before_n = len(df_data)
df_data = df_data[df_data["location_matched_clean"].apply(lambda x: len(x) > 0)].reset_index(drop=True)
after_n = len(df_data)
print(f"🧹 Dropped {before_n - after_n} rows with empty/[] locations; kept {after_n}.")

# ---------- PHASE 2: datetime (TEXT-FIRST, fallback to metadata) ----------
time_mentions_raw, time_parsed, event_time_primary_iso = [], [], []

for _, row in tqdm(df_data.iterrows(), total=len(df_data), desc="Phase 2: datetime"):
    text = str(row.get(TEXT_COL, "") or "")

    # text-first extraction
    mentions = extract_temporal_mentions(text)
    ref_raw  = _first_present(REF_COLS, row)
    ref_dt   = _parse_ref_dt(ref_raw)
    parsed   = normalize_temporal(text, ref_dt)

    # fallback ISO from metadata
    ref_dt_iso = ref_dt.isoformat() if isinstance(ref_dt, datetime) else None
    primary_iso = pick_primary_event_time(parsed, text, ref_dt_iso)

    time_mentions_raw.append(mentions)
    time_parsed.append(parsed)
    event_time_primary_iso.append(primary_iso)

df_data["time_mentions_raw"] = time_mentions_raw
df_data["time_parsed"] = time_parsed
df_data["event_time_primary_iso"] = event_time_primary_iso

# Pretty format for convenience
def format_event_time(iso_str):
    if not iso_str:
        return None
    try:
        dt = datetime.fromisoformat(iso_str)
        return dt.strftime("%B %d, %Y %H:%M:%S")
    except Exception:
        return iso_str

df_data["event_time_primary"] = df_data["event_time_primary_iso"].apply(format_event_time)

### Save and Preview

In [ ]:
import json
def clean_matches(matches):
    """
    Keep entities unless BOTH level and code are None.
    Remove duplicates (by level+code).
    """
    seen = set()
    cleaned = []
    for m in matches or []:
        if not m:
            continue
        # Drop only if both level and code are None
        if m.get("level") is None and m.get("code") is None:
            continue
        key = (m.get("level"), m.get("code"))
        if key not in seen:
            seen.add(key)
            cleaned.append(m)
    return cleaned

df_data["location_matched_clean"] = df_data["location_matched"].apply(clean_matches)

# 3) JSON-safe for CSV
df_data["location_matched_json"] = df_data["location_matched_clean"].apply(lambda x: json.dumps(x or [], ensure_ascii=False))
df_data["time_parsed_json"] = df_data["time_parsed"].apply(lambda x: json.dumps(x or [], ensure_ascii=False))
df_data["time_mentions_raw_json"] = df_data["time_mentions_raw"].apply(lambda x: json.dumps(x or [], ensure_ascii=False))


# 4) Custom columns
wanted = ["source", "query_page", "text", "post_url", "location_raw", "location_matched_json", "time_mentions_raw_json", "time_parsed_json", "event_time_primary"]

# Ensure requested columns exist
for col in wanted:
    if col not in df_data.columns:
        df_data[col] = ""

# Make lists printable in CSV
df_data["location_raw"] = df_data["location_raw"].apply(
    lambda x: json.dumps(x or [], ensure_ascii=False)
)

out_df = df_data[wanted].copy()

# 5) Save
out_dir = os.path.join(base_dir, "data", "processed-data")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "raw_data_with_location_time.csv")
out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"✅ Saved cleaned minimal dataset: {out_path}")